# EXTRACTION OF RASTERS BY AREA

- Raster reading: Opened only once in read mode.

- By zone: Loading of the 0.01° grid corresponding to the zone.

- By cell: For each 0.01° polygon, the average of the raster pixels inside is calculated.

- Precise masking: We use geometry_mask to count only the pixels in the cell.

- Result: Centre coordinates + average, saved in a CSV file per zone.

- RAM management: We only read the small window of the raster corresponding to each polygon, avoiding complete reading.

In [ ]:
# installing the python library to read and interact with rasters
pip install rasterio

In [ ]:
# importing all libraries needed for the extraction
import geopandas as gpd
import rasterio
import rasterio.features
import numpy as np
import pandas as pd
import os
from shapely.geometry import box
from tqdm import tqdm
import re
import zipfile

In [ ]:
# connection to the drive where the rasters are kept and where the results will be saved
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Here the parameters for the code to run smoothly are defined
# this code will loop on all the grids saved in a zipped folder (less place used and still efficient)

# === PARAMETERS ===
# path in the drive to the raster
RASTER_PATH = "/content/drive/MyDrive/Folder/raster.tif"
# path to the folder where you want the results to be saved as .csv in your drive
OUTPUT_FOLDER = "/content/drive/MyDrive/output_folder"
# Path to the zip file containing the grids of the different zones
ZIP_PATH = "/content/drive/MyDrive/Zone/grid_001_zone9.zip"
# Precise the raster's value here, as for it to be saved in the csv as column header as well as in the csv name
VALUE = "elevation"

# if the output folder does not exist, it is created
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [ ]:
# === This definition calculates the mean of the raster's values in the selected grid's cell ===
def raster_mean_in_polygon(src, polygon):
    """Calculate raster's mean value in a polygon."""
    # Check if the polygon intersects with the raster bounds
    if not polygon.intersects(box(*src.bounds)):
        return np.nan # Polygon is outside raster extent

    # Calculate the raster's window in the polygon
    # Use boundless=True to handle cases where the polygon is partially outside the raster
    window = rasterio.features.geometry_window(src, [polygon], pad_x=0, pad_y=0, boundless=True)

    # Read the sub-matrix raster corresponding to the window
    # Use masked=True to mask out areas outside the raster bounds when boundless=True
    data = src.read(1, window=window, masked=True)

    # Create a binary mask (true if in polygon)
    mask = rasterio.features.geometry_mask([polygon], transform=src.window_transform(window), invert=False, out_shape=data.shape)

    # Ensure the mask aligns with the data's shape when boundless=True
    # Use the mask to mask the data directly
    masked_data = np.ma.masked_array(data, mask=mask)

    # Check if there are any unmasked pixels before calculating the mean
    if masked_data.compressed().size == 0:
         return np.nan  # No valid data in the cell
    else:
        return masked_data.mean()

In [ ]:
# === Reading the raster and looping on all the grid's cells
print("⏳ Opening the raster...")

with rasterio.open(RASTER_PATH) as src:

    # Search the zip file
    with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
        # Only choose the files in format .gpkg
        gpkg_files = [f for f in archive.namelist() if f.endswith(".gpkg")]

        # Loop on all the grids
        for filename in gpkg_files:

            # Extract the zone's number from the name of the file
            match = re.search(r"grid_001_zone(\d+)", filename) # rename the search depending on the filename format
            if match:
                zone_id = int(match.group(1))
            else:
                print(f"Impossible to find the zone's number in this file : {filename}")
                continue

            print(f"\n➡️ Extraction zone_id {zone_id}")

            # Read the 0.01 grid
            grid_path =  f"/vsizip/{ZIP_PATH}/{filename}"
            grid_gdf = gpd.read_file(grid_path).reset_index(drop=True)

            # create the final list
            results = []

            # loop on all the cells of the grid with a track on progress (tqdm)
            for idx, cell in tqdm(grid_gdf.iterrows(), total=len(grid_gdf), desc=f"Zone {zone_id} extraction"):
                # get the geometry of the cell
                polygon = cell.geometry

                # Calculate the mean
                mean_val = raster_mean_in_polygon(src, polygon)

                # Get the coordinate of the cell's center
                centroid = polygon.centroid
                # copy the results in the final list
                results.append({
                    "longitude": round(centroid.x, 3),
                    "latitude": round(centroid.y, 3),
                    f"{VALUE}": mean_val
                })

            # create a dataframe with the results
            df_zone = pd.DataFrame(results)

            # Save as .csv
            try:
                # create a double entry matrix based on lat and long
                pivoted = df_zone.pivot(index="latitude", columns="longitude", values=f"{VALUE}")
                # keep the lat and long in the right order (N-S and W-E)
                pivoted = pivoted.sort_index(ascending=False)
                # save as csv
                OUTPUT_CSV = os.path.join(OUTPUT_FOLDER, f"Pivot_{VALUE}_zone_{zone_id}.csv")
                pivoted.to_csv(OUTPUT_CSV)
                print(f"✅ Double entry table saved at : {OUTPUT_CSV}")
            except Exception as e:
                # in case there was a problem to create the double entry table, the dataframe will be saved as is
                output_csv = os.path.join(OUTPUT_FOLDER, f"Extraction_{VALUE}_zone_{zone_id}.csv")
                df_zone.to_csv(output_csv, index=False)
                print(f"❌ Double entry was not created : {e}\n✅ Data still saved in brut at {output_csv}.")